# Gaze-only baseline — TEyeD Dikablis

Model predviđa 3D jedinični vektor pogleda. Ne učitava segmentacijske maske, ne koristi `mask_valid_*` i nema segmentacijsku granu.

**Eksperimentalna kontrola:** isti subject-disjoint train/val/test split iz `Preprocessed.ipynb`; isti broj seedova i zajednički parametri definirani su u konfiguracijskoj ćeliji. Augmentacija je implementirana nad slikom, uz odgovarajuću promjenu predznaka X komponente kod horizontalnog flip-a i rotaciju XY komponenti kod rotacije slike. Ako MTL implementacija koristi drukčije augmentacijske magnitude, uskladi konstante u konfiguracijskoj ćeliji prije usporedbe.


In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import gc
import json
import math
import random
import time
import warnings
import subprocess
import traceback
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# TensorFlow Lite opcionalno pokušava učitati JAX.
# Instalirana JAX/SciPy kombinacija nije kompatibilna, a JAX nam ne treba.
sys.modules["jax"] = None

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_addons as tfa
import matplotlib.pyplot as plt

print(f"Python interpreter: {sys.executable}")
print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__ if hasattr(keras, '__version__') else 'integrated'}")
print(f"TensorFlow Addons: {tfa.__version__}")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")

# Provjera kompatibilnosti okruženja.
if tf.__version__ != "2.11.0":
    warnings.warn(
        f"Očekivan je TensorFlow 2.11.0, pronađen je {tf.__version__}."
    )

if tfa.__version__ != "0.19.0":
    warnings.warn(
        f"Očekivan je TensorFlow Addons 0.19.0, pronađen je {tfa.__version__}."
    )

try:
    import numexpr
    print(f"numexpr: {numexpr.__version__}")
except ImportError:
    print("numexpr nije instaliran.")

try:
    import bottleneck
    print(f"bottleneck: {bottleneck.__version__}")
except ImportError:
    print("bottleneck nije instaliran.")


# ---------------------------------------------------------
# Putanje i osnovna konfiguracija
# ---------------------------------------------------------

SEEDS = [42, 123, 2024, 7, 13]

BASE_PATH = Path("/home/fipulab/anastazija/TEyeD_preprocessed")
DATASET_FOLDER = "128x128_stride10_q4_seg_pupil-iris-lid_2D"

PREPROCESSED_ROOT = BASE_PATH / DATASET_FOLDER

SPLIT_ROOTS = {
    split: PREPROCESSED_ROOT / split
    for split in ("train", "val", "test")
}

OUTPUT_BASE = Path("/home/fipulab/anastazija/runs/gaze_only")

IMAGE_SIZE = (128, 128)

BATCH_SIZE = 128
BATCH_FALLBACKS = [128, 64, 32]

PHASE1_EPOCHS = 6
PHASE2_EPOCHS = 60

PHASE1_LR = 1e-3
PHASE2_LR = 1e-4

WEIGHT_DECAY = 1e-3

DROPOUT = 0.3
DENSE_UNITS = 256

NUM_PARALLEL_CALLS = 4
AUTOTUNE = tf.data.AUTOTUNE

MAX_SEED_HOURS = 6


# ---------------------------------------------------------
# Augmentacija
# ---------------------------------------------------------

HORIZONTAL_FLIP = True

ROTATION_FACTOR = 0.025
ZOOM_FACTOR = 0.05
SHEAR_FACTOR = 0.02

BRIGHTNESS_DELTA = 0.08

CONTRAST_LOWER = 0.90
CONTRAST_UPPER = 1.10


# ---------------------------------------------------------
# TensorFlow threading
# ---------------------------------------------------------

try:
    tf.config.threading.set_inter_op_parallelism_threads(4)
    tf.config.threading.set_intra_op_parallelism_threads(4)
except RuntimeError as exc:
    warnings.warn(
        f"TensorFlow threading već je inicijaliziran: {exc}"
    )


# ---------------------------------------------------------
# GPU i mixed precision
# ---------------------------------------------------------

GPUS = tf.config.list_physical_devices("GPU")

if GPUS:
    for gpu in GPUS:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as exc:
            warnings.warn(
                f"Ne mogu postaviti GPU memory growth: {exc}"
            )

    try:
        keras.mixed_precision.set_global_policy("mixed_float16")
        print("GPU dostupan: mixed_float16 uključen.")
    except Exception as exc:
        keras.mixed_precision.set_global_policy("float32")
        warnings.warn(
            f"Mixed precision nije uključen: {exc}"
        )
else:
    keras.mixed_precision.set_global_policy("float32")
    warnings.warn(
        "GPU nije dostupan; treniranje će koristiti CPU i float32."
    )

print(
    "Mixed precision policy:",
    keras.mixed_precision.global_policy()
)

print(
    "Legacy Adam dostupan:",
    hasattr(keras.optimizers, "legacy")
    and hasattr(keras.optimizers.legacy, "Adam")
)


# ---------------------------------------------------------
# GPU memorija
# ---------------------------------------------------------

def gpu_memory_report():
    if not GPUS:
        print("GPU memorija: GPU nije dostupan.")
        return None

    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=memory.total,memory.used,memory.free",
                "--format=csv,noheader,nounits"
            ],
            text=True
        ).strip().splitlines()

        free = []

        for i, line in enumerate(out):
            total, used, available = [
                float(v.strip()) for v in line.split(",")
            ]

            free_gb = available / 1024
            free.append(free_gb)

            print(
                f"GPU {i}: "
                f"ukupno {total / 1024:.2f} GB, "
                f"zauzeto {used / 1024:.2f} GB, "
                f"slobodno {free_gb:.2f} GB"
            )

        if min(free) < 4:
            warnings.warn(
                "Slobodna GPU memorija manja je od 4 GB."
            )

        return min(free)

    except Exception as exc:
        warnings.warn(
            f"nvidia-smi nije dostupan: {exc}"
        )
        return None


gpu_memory_report()


# ---------------------------------------------------------
# Provjera dataset CSV datoteka
# ---------------------------------------------------------

SPLIT_DF = {}

required = {
    "filename",
    "x",
    "y",
    "z",
    "subject_id",
    "video_id"
}

for split, root in SPLIT_ROOTS.items():

    csv_path = root / "labels.csv"

    if not csv_path.exists():
        raise FileNotFoundError(
            f"Nedostaje {csv_path}"
        )

    df = pd.read_csv(csv_path)

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{csv_path}: nedostaju stupci {sorted(missing)}"
        )

    # Koristimo samo gaze oznake.
    # mask_valid_* stupci se namjerno ignoriraju.
    for column in ("x", "y", "z"):
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

    gaze_values = df[["x", "y", "z"]].to_numpy(
        dtype=np.float32
    )

    norms = np.linalg.norm(
        gaze_values,
        axis=1
    )

    good = (
        np.isfinite(norms)
        & (norms > 0)
        & np.isfinite(gaze_values).all(axis=1)
    )

    if not np.all(good):

        warnings.warn(
            f"{split}: izbacujem "
            f"{int((~good).sum())} redaka s nevaljanim "
            "ili nul-vektor gaze oznakama."
        )

        df = df.loc[good].copy()

        vectors = df[["x", "y", "z"]].to_numpy(
            dtype=np.float32
        )

        vectors /= np.linalg.norm(
            vectors,
            axis=1,
            keepdims=True
        )

        df[["x", "y", "z"]] = vectors

    SPLIT_DF[split] = df.reset_index(drop=True)

    print(
        f"{split}: {len(df):,} uzoraka, "
        f"{df.video_id.nunique():,} videa, "
        f"{df.subject_id.nunique():,} ispitanika"
    )


# ---------------------------------------------------------
# Provjera subject-disjoint podjele
# ---------------------------------------------------------

subject_sets = {
    split: set(df.subject_id.astype(str))
    for split, df in SPLIT_DF.items()
}

for a, b in (
    ("train", "val"),
    ("train", "test"),
    ("val", "test")
):

    overlap = subject_sets[a] & subject_sets[b]

    if overlap:
        raise ValueError(
            f"Ispitanici se preklapaju između {a} i {b}: "
            f"{sorted(overlap)[:10]}"
        )

print("Provjera subject-disjoint splitova: OK.")

OUTPUT_BASE.mkdir(
    parents=True,
    exist_ok=True
)

Python interpreter: /home/fipulab/anaconda3/envs/anastazija_env/bin/python
TensorFlow: 2.11.0
Keras: 2.11.0
TensorFlow Addons: 0.19.0
NumPy: 1.24.3 | Pandas: 2.0.3
numexpr nije instaliran.
bottleneck nije instaliran.
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3090, compute capability 8.6
GPU dostupan: mixed_float16 uključen.
Mixed precision policy: <Policy "mixed_float16">
Legacy Adam dostupan: True
GPU 0: ukupno 23.70 GB, zauzeto 0.02 GB, slobodno 23.68 GB
train: 339,466 uzoraka, 264 videa, 76 ispitanika
val: 135,958 uzoraka, 108 videa, 22 ispitanika
test: 79,964 uzoraka, 50 videa, 11 ispitanika
Provjera subject-disjoint splitova: OK.


## 2. tf.data i augmentacija

Slike se čitaju lijeno iz `train/<video_id>/<frame>.jpg`. Ne učitavaju se maske. Veliki skupovi se ne cacheiraju u RAM-u. Horizontalni flip mijenja predznak `x`; rotacija slike oko središta primjenjuje istu rotaciju na `x,y` gaze komponente.


In [2]:
def set_all_seeds(seed):

    random.seed(seed)
    np.random.seed(seed)

    tf.random.set_seed(seed)

    tf.keras.utils.set_random_seed(seed)


def _load_image(filename, split_root):

    path = tf.strings.join([
        str(split_root),
        "/",
        filename
    ])

    image = tf.io.decode_jpeg(
        tf.io.read_file(path),
        channels=3
    )

    image = tf.image.resize(
        image,
        IMAGE_SIZE,
        method="bilinear"
    )

    image = tf.cast(
        image,
        tf.float32
    ) / 127.5 - 1.0

    image.set_shape((128, 128, 3))

    return image


def _augment_pair(image, gaze):

    # -----------------------------------------------------
    # Horizontal flip
    # -----------------------------------------------------

    if HORIZONTAL_FLIP:

        do_flip = tf.random.uniform(()) < 0.5

        image = tf.cond(
            do_flip,
            lambda: tf.image.flip_left_right(image),
            lambda: image
        )

        gaze = tf.cond(
            do_flip,
            lambda: tf.stack([
                -gaze[0],
                gaze[1],
                gaze[2]
            ]),
            lambda: gaze
        )


    # -----------------------------------------------------
    # Rotation
    # -----------------------------------------------------

    angle = tf.random.uniform(
        (),
        -ROTATION_FACTOR,
        ROTATION_FACTOR
    ) * (2.0 * math.pi)

    c = tf.cos(angle)
    s = tf.sin(angle)

    cx = 63.5
    cy = 63.5

    a0 = c
    a1 = s

    b0 = -s
    b1 = c

    tx = cx - a0 * cx - a1 * cy
    ty = cy - b0 * cx - b1 * cy

    transform = tf.stack([
        a0, a1, tx,
        b0, b1, ty,
        0.0, 0.0
    ])[None, :]

    image = tf.raw_ops.ImageProjectiveTransformV3(
        images=image[None, ...],
        transforms=transform,
        output_shape=tf.constant([128, 128]),
        interpolation="BILINEAR",
        fill_mode="REFLECT",
        fill_value=0.0
    )[0]

    gx = c * gaze[0] - s * gaze[1]
    gy = s * gaze[0] + c * gaze[1]

    gaze = tf.stack([
        gx,
        gy,
        gaze[2]
    ])

    gaze = tf.math.l2_normalize(gaze)


    # -----------------------------------------------------
    # Shear
    # -----------------------------------------------------

    shear = tf.random.uniform(
        (),
        -SHEAR_FACTOR,
        SHEAR_FACTOR
    )

    shear_transform = tf.stack([
        1.0,
        shear,
        -shear * 63.5,
        0.0,
        1.0,
        0.0,
        0.0,
        0.0
    ])[None, :]

    image = tf.raw_ops.ImageProjectiveTransformV3(
        images=image[None, ...],
        transforms=shear_transform,
        output_shape=tf.constant([128, 128]),
        interpolation="BILINEAR",
        fill_mode="REFLECT",
        fill_value=0.0
    )[0]

    gaze = tf.math.l2_normalize(
        tf.stack([
            gaze[0] + shear * gaze[1],
            gaze[1],
            gaze[2]
        ])
    )


    # -----------------------------------------------------
    # Zoom
    # -----------------------------------------------------

    zoom = tf.random.uniform(
        (),
        1.0 - ZOOM_FACTOR,
        1.0 + ZOOM_FACTOR
    )

    zx = 1.0 / zoom
    ztx = 63.5 - zx * 63.5

    zform = tf.stack([
        zx, 0.0, ztx,
        0.0, zx, ztx,
        0.0, 0.0
    ])[None, :]

    image = tf.raw_ops.ImageProjectiveTransformV3(
        images=image[None, ...],
        transforms=zform,
        output_shape=tf.constant([128, 128]),
        interpolation="BILINEAR",
        fill_mode="REFLECT",
        fill_value=0.0
    )[0]


    # -----------------------------------------------------
    # Photometric augmentations
    # -----------------------------------------------------

    image = tf.image.random_brightness(
        image,
        max_delta=BRIGHTNESS_DELTA
    )

    image = tf.image.random_contrast(
        image,
        lower=CONTRAST_LOWER,
        upper=CONTRAST_UPPER
    )

    image = tf.clip_by_value(
        image,
        -1.0,
        1.0
    )

    return image, gaze


def make_dataset(
    df,
    split_root,
    batch_size,
    training=False,
    seed=42
):

    filenames = df["filename"].astype(str).to_numpy()

    gaze = df[["x", "y", "z"]].to_numpy(
        dtype=np.float32
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (filenames, gaze)
    )

    if training:

        ds = ds.shuffle(
            min(len(df), 20000),
            seed=seed,
            reshuffle_each_iteration=True
        )

    root = str(split_root)

    def load_map(filename, vector):

        image = _load_image(
            filename,
            root
        )

        vector = tf.math.l2_normalize(
            tf.cast(vector, tf.float32)
        )

        if training:
            image, vector = _augment_pair(
                image,
                vector
            )

        return image, vector

    ds = ds.map(
        load_map,
        num_parallel_calls=NUM_PARALLEL_CALLS,
        deterministic=not training
    )

    ds = ds.batch(
        batch_size,
        drop_remainder=False
    )

    # Bez cache() zbog veličine dataseta.
    return ds.prefetch(AUTOTUNE)

## 3. Model i metrike

Gubitak je kosinusna udaljenost `1 - dot(pred, gt)`. Metrika je kutna pogreška u stupnjevima. Završni Dense izlaz je float32 radi stabilnosti uz mixed precision.


In [3]:
def build_gaze_model():

    inputs = keras.Input(
        shape=(128, 128, 3),
        name="image"
    )

    encoder = keras.applications.MobileNetV3Small(
        input_shape=(128, 128, 3),
        include_top=False,
        weights="imagenet",
        include_preprocessing=False
    )

    x = encoder(inputs)

    x = layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = layers.Dense(
        DENSE_UNITS,
        activation="relu",
        name="gaze_dense"
    )(x)

    x = layers.Dropout(
        DROPOUT,
        name="gaze_dropout"
    )(x)

    raw = layers.Dense(
        3,
        dtype="float32",
        name="gaze_raw"
    )(x)

    output = layers.Lambda(
        lambda t: tf.math.l2_normalize(t, axis=-1),
        name="gaze_unit_vector",
        dtype="float32"
    )(raw)

    model = keras.Model(
        inputs,
        output,
        name="MobileNetV3Small_GazeOnly"
    )

    return model, encoder


# ---------------------------------------------------------
# Loss: 1 - cosine similarity
# ---------------------------------------------------------

def cosine_distance_loss(y_true, y_pred):

    y_true = tf.math.l2_normalize(
        tf.cast(y_true, tf.float32),
        axis=-1
    )

    y_pred = tf.math.l2_normalize(
        tf.cast(y_pred, tf.float32),
        axis=-1
    )

    dot = tf.reduce_sum(
        y_true * y_pred,
        axis=-1
    )

    dot = tf.clip_by_value(
        dot,
        -1.0,
        1.0
    )

    return 1.0 - dot


# ---------------------------------------------------------
# Mean angular error
# ---------------------------------------------------------

def mean_angle_error(y_true, y_pred):

    y_true = tf.math.l2_normalize(
        tf.cast(y_true, tf.float32),
        axis=-1
    )

    y_pred = tf.math.l2_normalize(
        tf.cast(y_pred, tf.float32),
        axis=-1
    )

    dot = tf.reduce_sum(
        y_true * y_pred,
        axis=-1
    )

    angle = tf.acos(
        tf.clip_by_value(dot, -1.0, 1.0)
    ) * (180.0 / math.pi)

    return tf.reduce_mean(angle)


mean_angle_error.__name__ = "mean_angle_error"


# ---------------------------------------------------------
# Optimizatori kompatibilni s TF/Keras 2.11
# ---------------------------------------------------------

def make_optimizer(lr, phase):

    if phase == 1:

        # Legacy Adam izbjegava problem s novim Keras
        # optimizer API-jem i mixed precision AutoCastVariable.
        return keras.optimizers.legacy.Adam(
            learning_rate=lr
        )

    elif phase == 2:

        # TensorFlow Addons 0.19.0 AdamW.
        return tfa.optimizers.AdamW(
            learning_rate=lr,
            weight_decay=WEIGHT_DECAY
        )

    else:

        raise ValueError(
            f"Nepoznata faza: {phase}"
        )


# ---------------------------------------------------------
# Kompilacija modela
# ---------------------------------------------------------

def compile_model(model, lr, phase):

    model.compile(
        optimizer=make_optimizer(lr, phase),
        loss=cosine_distance_loss,
        metrics=[mean_angle_error],
        jit_compile=False
    )


# ---------------------------------------------------------
# Zamrzavanje / odmrzavanje encodera
# ---------------------------------------------------------

def set_encoder_trainable(encoder, trainable):

    encoder.trainable = trainable

    if trainable:

        for layer in encoder.layers:

            layer.trainable = not isinstance(
                layer,
                layers.BatchNormalization
            )

    else:

        for layer in encoder.layers:
            layer.trainable = False

## 4. Callbackovi, evaluacija i grafovi

Checkpoint prati `val_mean_angle_error` (stupnjevi). Povijesti obiju faza spajaju se u jedan CSV; početak fine-tuninga označen je okomitom linijom.


In [4]:
class EpochTimer(keras.callbacks.Callback):

    def __init__(self, total_epochs):

        super().__init__()

        self.total_epochs = total_epochs
        self.started = None
        self.durations = []

    def on_epoch_begin(self, epoch, logs=None):

        self.started = time.time()

    def on_epoch_end(self, epoch, logs=None):

        elapsed = time.time() - self.started

        self.durations.append(elapsed)

        avg = float(
            np.mean(self.durations)
        )

        remaining = max(
            0,
            self.total_epochs - (epoch + 1)
        )

        print(
            f"Procjena vremena po epohi: "
            f"{avg / 60:.1f} min; "
            f"preostalo približno "
            f"{remaining * avg / 3600:.2f} h"
        )


def callbacks_for(output_dir, total_epochs):

    return [

        keras.callbacks.ModelCheckpoint(
            filepath=str(
                output_dir / "best_model.keras"
            ),
            monitor="val_mean_angle_error",
            mode="min",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),

        keras.callbacks.ReduceLROnPlateau(
            monitor="val_mean_angle_error",
            mode="min",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        ),

        keras.callbacks.EarlyStopping(
            monitor="val_mean_angle_error",
            mode="min",
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),

        EpochTimer(total_epochs)
    ]


def combine_histories(h1, h2):

    d1 = pd.DataFrame(h1.history)
    d2 = pd.DataFrame(h2.history)

    if len(d1) and len(d2):

        return pd.concat(
            [d1, d2],
            ignore_index=True
        )

    return d1 if len(d1) else d2


# ---------------------------------------------------------
# Test evaluacija
# ---------------------------------------------------------

def evaluate_test(
    model,
    df,
    split_root,
    batch_size,
    seed,
    output_dir
):

    ds = make_dataset(
        df,
        split_root,
        batch_size,
        training=False
    )

    rows = []
    offset = 0

    for images, targets in ds:

        preds = model(
            images,
            training=False
        ).numpy().astype(np.float32)

        gts = targets.numpy().astype(np.float32)

        batch_n = len(preds)

        batch_meta = df.iloc[
            offset:offset + batch_n
        ]

        assert batch_n == len(batch_meta), (
            f"Batch/metadata mismatch: "
            f"{batch_n} != {len(batch_meta)}"
        )

        offset += batch_n

        norms = np.linalg.norm(
            preds,
            axis=1
        )

        for i in range(batch_n):

            if (
                not np.isfinite(norms[i])
                or norms[i] <= 0
            ):

                warnings.warn(
                    f"Seed {seed}: preskačem predikciju "
                    f"s normom 0/nevaljanom normom: "
                    f"{batch_meta.iloc[i]['filename']}"
                )

                continue

            pred = preds[i] / norms[i]

            gt = gts[i]

            dot = float(
                np.clip(
                    np.dot(pred, gt),
                    -1.0,
                    1.0
                )
            )

            angle = float(
                np.degrees(np.arccos(dot))
            )

            meta = batch_meta.iloc[i]

            rows.append({

                "subject_id": str(meta["subject_id"]),
                "video_id": str(meta["video_id"]),
                "filename": str(meta["filename"]),

                "gt_x": float(gt[0]),
                "gt_y": float(gt[1]),
                "gt_z": float(gt[2]),

                "pred_x": float(pred[0]),
                "pred_y": float(pred[1]),
                "pred_z": float(pred[2]),

                "angle_deg": angle
            })


    pred_df = pd.DataFrame(
        rows,
        columns=[
            "subject_id",
            "video_id",
            "filename",
            "gt_x",
            "gt_y",
            "gt_z",
            "pred_x",
            "pred_y",
            "pred_z",
            "angle_deg"
        ]
    )

    if pred_df.empty:

        raise RuntimeError(
            f"Seed {seed}: nema valjanih test predikcija."
        )

    pred_df.to_csv(
        output_dir / "test_predictions.csv",
        index=False
    )

    per_subject = (
        pred_df
        .groupby("subject_id")["angle_deg"]
        .mean()
        .to_dict()
    )

    result = {

        "seed": int(seed),

        "mean_angle_error": float(
            pred_df.angle_deg.mean()
        ),

        "std_angle_error": float(
            pred_df.angle_deg.std(ddof=0)
        ),

        "num_subjects": int(
            pred_df.subject_id.nunique()
        ),

        "per_subject_mean_angle_error": {
            str(k): float(v)
            for k, v in per_subject.items()
        },

        "n_test_samples": int(
            len(pred_df)
        )
    }

    return result, pred_df


# ---------------------------------------------------------
# Vizualizacije
# ---------------------------------------------------------

def save_figures(
    history_df,
    pred_df,
    figures_dir,
    seed
):

    figures_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    if not history_df.empty:

        x = np.arange(
            1,
            len(history_df) + 1
        )

        for key, title, ylabel in [

            (
                "loss",
                "Gaze loss",
                "Cosine distance loss"
            ),

            (
                "mean_angle_error",
                "Mean angular error",
                "Mean angle error (degrees)"
            )

        ]:

            if (
                key not in history_df
                or "val_" + key not in history_df
            ):
                continue

            plt.figure(
                figsize=(9, 5),
                dpi=160
            )

            plt.plot(
                x,
                history_df[key],
                label="Train"
            )

            plt.plot(
                x,
                history_df["val_" + key],
                label="Validation"
            )

            if len(history_df) > PHASE1_EPOCHS:

                plt.axvline(
                    PHASE1_EPOCHS + 0.5,
                    linestyle="--",
                    label="Full fine-tuning starts"
                )

            plt.xlabel("Epoch")
            plt.ylabel(ylabel)

            plt.title(
                f"{title} — seed {seed}"
            )

            plt.grid(
                True,
                alpha=0.3
            )

            plt.legend()
            plt.tight_layout()

            plt.savefig(
                figures_dir / f"seed_{seed}_{key}.png"
            )

            plt.close()


    plt.figure(
        figsize=(8, 5),
        dpi=160
    )

    plt.hist(
        pred_df["angle_deg"],
        bins=40
    )

    plt.xlabel("Angular error (degrees)")
    plt.ylabel("Number of samples")

    plt.title(
        f"Test angular error — seed {seed}"
    )

    plt.tight_layout()

    plt.savefig(
        figures_dir / f"seed_{seed}_test_angle_histogram.png"
    )

    plt.close()

## 5. Treniranje pet seedova i siguran nastavak

Ako postoji `best_model.keras`, seed se ne trenira ponovno nego se model učitava i evaluira. Ako dođe do OOM-a, batch se smanjuje 128 → 64 → 32. Svaki seed ima vlastiti `try/except`; povijest i konfiguracija spremaju se nakon svakog seeda, a djelomična povijest čuva se i pri prekidu.


In [5]:
def train_one_seed(seed):

    seed = int(seed)

    set_all_seeds(seed)

    out = OUTPUT_BASE / f"seed_{seed}"

    out.mkdir(
        parents=True,
        exist_ok=True
    )

    model_path = out / "best_model.keras"
    history_path = out / "history.csv"
    config_path = out / "config.json"
    summary_path = out / "summary.json"

    started = time.time()

    config = {

        "seed": seed,

        "batch_size_requested": BATCH_SIZE,

        "phase1_epochs": PHASE1_EPOCHS,
        "phase2_epochs": PHASE2_EPOCHS,

        "phase1_lr": PHASE1_LR,
        "phase2_lr": PHASE2_LR,

        "weight_decay": WEIGHT_DECAY,

        "dense_units": DENSE_UNITS,
        "dropout": DROPOUT,

        "optimizer": (
            "Keras legacy Adam (phase 1) + "
            "TensorFlow Addons AdamW (phase 2)"
        ),

        "loss": "1 - cosine_similarity",

        "architecture": (
            "MobileNetV3Small + GAP + Dense(256,relu) "
            "+ Dropout(0.3) + Dense(3) + L2 normalize"
        ),

        "tensorflow_version": tf.__version__,
        "tensorflow_addons_version": tfa.__version__,

        "mixed_precision": bool(GPUS),

        "augmentation": {

            "horizontal_flip": HORIZONTAL_FLIP,

            "rotation_factor": ROTATION_FACTOR,

            "zoom_factor": ZOOM_FACTOR,

            "shear_factor": SHEAR_FACTOR,

            "brightness_delta": BRIGHTNESS_DELTA,

            "contrast": [
                CONTRAST_LOWER,
                CONTRAST_UPPER
            ]
        },

        "dataset": str(PREPROCESSED_ROOT),

        "started_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    }


    # -----------------------------------------------------
    # Ako je seed već dovršen, preskoči treniranje
    # -----------------------------------------------------

    if model_path.exists() and summary_path.exists():

        print(
            f"seed {seed} već je dovršen, "
            "preskačem treniranje i radim evaluaciju."
        )

        model = keras.models.load_model(
            model_path,
            compile=False
        )

        batch = BATCH_SIZE

        result, pred_df = evaluate_test(
            model,
            SPLIT_DF["test"],
            SPLIT_ROOTS["test"],
            batch,
            seed,
            out
        )

        save_figures(
            pd.read_csv(history_path)
            if history_path.exists()
            else pd.DataFrame(),

            pred_df,

            OUTPUT_BASE / "figures",

            seed
        )

        result["config"] = config

        summary_path.write_text(
            json.dumps(result, indent=2),
            encoding="utf-8"
        )

        config["elapsed_seconds"] = (
            time.time() - started
        )

        config["status"] = "loaded_existing_model"

        config_path.write_text(
            json.dumps(config, indent=2),
            encoding="utf-8"
        )

        return result


    # -----------------------------------------------------
    # Pokušaji s batch size vrijednostima 128, 64 i 32
    # -----------------------------------------------------

    last_error = None

    batch_sizes = [
        value
        for value in BATCH_FALLBACKS
        if value <= BATCH_SIZE
    ]

    for batch in batch_sizes:

        try:

            tf.keras.backend.clear_session()
            gc.collect()

            set_all_seeds(seed)

            train_ds = make_dataset(
                SPLIT_DF["train"],
                SPLIT_ROOTS["train"],
                batch,
                training=True,
                seed=seed
            )

            val_ds = make_dataset(
                SPLIT_DF["val"],
                SPLIT_ROOTS["val"],
                batch,
                training=False,
                seed=seed
            )

            model, encoder = build_gaze_model()

            # Postoji checkpoint, ali nema summary.json:
            # nastavljamo iz spremljenih težina.
            resumed_from_phase1 = (
                model_path.exists()
                and not summary_path.exists()
            )


            # -------------------------------------------------
            # Nastavak iz checkpointa
            # -------------------------------------------------

            if resumed_from_phase1:

                print(
                    f"=== seed {seed}: pronađen checkpoint; "
                    "učitavam spremljene težine i nastavljam "
                    "iz faze 2 ==="
                )

                checkpoint_model = keras.models.load_model(
                    model_path,
                    compile=False
                )

                model.set_weights(
                    checkpoint_model.get_weights()
                )

                del checkpoint_model

                h1 = None


            # -------------------------------------------------
            # Faza 1: encoder frozen
            # -------------------------------------------------

            else:

                set_encoder_trainable(
                    encoder,
                    False
                )

                compile_model(
                    model,
                    PHASE1_LR,
                    phase=1
                )

                print(
                    f"=== seed {seed}: faza 1, "
                    f"encoder frozen, batch={batch} ==="
                )

                h1 = model.fit(

                    train_ds,

                    validation_data=val_ds,

                    epochs=PHASE1_EPOCHS,

                    callbacks=callbacks_for(
                        out,
                        PHASE1_EPOCHS + PHASE2_EPOCHS
                    ),

                    verbose=1
                )


            # -------------------------------------------------
            # Faza 2: full fine-tuning
            # BatchNormalization slojevi ostaju zamrznuti.
            # -------------------------------------------------

            set_encoder_trainable(
                encoder,
                True
            )

            compile_model(
                model,
                PHASE2_LR,
                phase=2
            )

            print(
                f"=== seed {seed}: faza 2, "
                f"full fine-tune, batch={batch} ==="
            )

            phase2_initial_epoch = (
                PHASE1_EPOCHS
                if resumed_from_phase1
                else len(h1.epoch)
            )

            h2 = model.fit(

                train_ds,

                validation_data=val_ds,

                initial_epoch=phase2_initial_epoch,

                epochs=PHASE1_EPOCHS + PHASE2_EPOCHS,

                callbacks=callbacks_for(
                    out,
                    PHASE1_EPOCHS + PHASE2_EPOCHS
                ),

                verbose=1
            )


            # -------------------------------------------------
            # Spajanje povijesti
            # -------------------------------------------------

            if resumed_from_phase1:

                # Povijest prve faze nije dostupna ako je
                # notebook ranije prekinut nakon checkpointa.
                hist = pd.DataFrame(h2.history)

            else:

                hist = combine_histories(
                    h1,
                    h2
                )

            hist.to_csv(
                history_path,
                index=False
            )


            # -------------------------------------------------
            # Spremanje modela
            # -------------------------------------------------

            # ModelCheckpoint tijekom faze 2 sprema najbolji
            # validation checkpoint u best_model.keras.
            #
            # Ako checkpoint ne postoji, spremi trenutačni model.
            if not model_path.exists():

                model.save(
                    model_path
                )


            # Učitaj najbolji spremljeni checkpoint.
            model = keras.models.load_model(
                model_path,
                compile=False
            )


            # -------------------------------------------------
            # Test evaluacija
            # -------------------------------------------------

            result, pred_df = evaluate_test(

                model,

                SPLIT_DF["test"],

                SPLIT_ROOTS["test"],

                batch,

                seed,

                out
            )


            # -------------------------------------------------
            # Vizualizacije
            # -------------------------------------------------

            save_figures(

                hist,

                pred_df,

                OUTPUT_BASE / "figures",

                seed
            )


            # -------------------------------------------------
            # Zapis konfiguracije i rezultata
            # -------------------------------------------------

            elapsed = time.time() - started

            config.update({

                "batch_size_used": batch,

                "elapsed_seconds": elapsed,

                "elapsed_hours": elapsed / 3600,

                "status": "completed",

                "phase1_epochs_completed": (

                    PHASE1_EPOCHS
                    if resumed_from_phase1
                    else len(h1.epoch)
                ),

                "phase2_epochs_completed": len(h2.epoch)
            })


            if elapsed > MAX_SEED_HOURS * 3600:

                warnings.warn(
                    f"Seed {seed} trajao je više od 6 sati."
                )


            result["config"] = config

            summary_path.write_text(
                json.dumps(result, indent=2),
                encoding="utf-8"
            )

            config_path.write_text(
                json.dumps(config, indent=2),
                encoding="utf-8"
            )


            print(
                f"Seed {seed} završen za "
                f"{elapsed / 3600:.2f} h; "
                f"test mean angle error="
                f"{result['mean_angle_error']:.3f}°"
            )

            return result


        # -----------------------------------------------------
        # OOM: pokušaj s manjim batch sizeom
        # -----------------------------------------------------

        except (
            tf.errors.ResourceExhaustedError,
            MemoryError
        ) as exc:

            last_error = exc

            warnings.warn(
                f"OOM za seed {seed}, batch={batch}: {exc}"
            )

            gc.collect()
            tf.keras.backend.clear_session()

            print(
                f"Batch size {batch} nije moguće koristiti."
            )

            continue


        # -----------------------------------------------------
        # Prekid korisnika
        # -----------------------------------------------------

        except KeyboardInterrupt:

            config.update({

                "status": "interrupted",

                "elapsed_seconds": (
                    time.time() - started
                ),

                "batch_size_used": batch
            })

            config_path.write_text(
                json.dumps(config, indent=2),
                encoding="utf-8"
            )

            warnings.warn(
                f"Seed {seed} prekinut. "
                "Ako postoji best_model.keras bez summary.json, "
                "sljedeće pokretanje pokušat će nastaviti "
                "iz spremljenog checkpointa."
            )

            raise


        finally:

            gc.collect()


    if last_error is not None:

        raise last_error

    raise RuntimeError(
        f"Seed {seed} nije dovršen."
    )


# ---------------------------------------------------------
# Pokretanje svih seedova
# ---------------------------------------------------------

seed_results = {}
seed_errors = {}

for seed in SEEDS:

    try:

        result = train_one_seed(seed)

        seed_results[int(seed)] = result

    except KeyboardInterrupt:

        print(
            "Prekid korisnika (Ctrl+C). "
            "Petlja je zaustavljena."
        )

        break

    except Exception as exc:

        seed_errors[int(seed)] = repr(exc)

        print(
            f"GREŠKA za seed {seed}: "
            f"{type(exc).__name__}: {exc}"
        )

        traceback.print_exc()

        gc.collect()

        tf.keras.backend.clear_session()


print(
    "Uspješni seedovi:",
    sorted(seed_results)
)

print(
    "Seedovi s greškom:",
    seed_errors
)

Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089
=== seed 42: faza 1, encoder frozen, batch=128 ===
Epoch 1/6
2652/2653 [============================>.] - ETA: 0s - loss: 0.0083 - mean_angle_error: 5.5053
Epoch 1: val_mean_angle_error improved from inf to 5.05773, saving model to /home/fipulab/anastazija/runs/gaze_only/seed_42/best_model.keras
Procjena vremena po epohi: 3.5 min; preostalo približno 3.79 h
2653/2653 [==============================] - 210s 78ms/step - loss: 0.0083 - mean_angle_error: 5.5049 - val_loss: 0.0062 - val_mean_angle_error: 5.0577 - lr: 0.0010
Epoch 2/6
2652/2653 [============================>.] - ETA: 0s - loss: 0.0066 - mean_angle_error: 4.8032
Epoch 2: val_mean_angle_error improved from 5.05773 to 4.94051, saving model to /home/fipulab/anastazija/runs/gaze_only/seed_42/best_model.keras
Procjena vremena po epoh

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



2652/2653 [============================>.] - ETA: 0s - loss: 0.0102 - mean_angle_error: 6.4812
Epoch 17: val_mean_angle_error improved from 5.95680 to 5.94886, saving model to /home/fipulab/anastazija/runs/gaze_only/seed_42/best_model.keras
Procjena vremena po epohi: 3.4 min; preostalo približno 2.74 h
2653/2653 [==============================] - 202s 76ms/step - loss: 0.0102 - mean_angle_error: 6.4801 - val_loss: 0.0083 - val_mean_angle_error: 5.9489 - lr: 5.0000e-05
Epoch 18/66
2652/2653 [============================>.] - ETA: 0s - loss: 0.0101 - mean_angle_error: 6.4818
Epoch 18: val_mean_angle_error did not improve from 5.94886
Procjena vremena po epohi: 3.4 min; preostalo približno 2.69 h
2653/2653 [==============================] - 207s 78ms/step - loss: 0.0101 - mean_angle_error: 6.4815 - val_loss: 0.0086 - val_mean_angle_error: 6.1534 - lr: 5.0000e-05
Epoch 19/66
2652/2653 [============================>.] - ETA: 0s - loss: 0.0102 - mean_angle_error: 6.4831
Epoch 19: val_mean_an

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



2652/2653 [============================>.] - ETA: 0s - loss: 0.0101 - mean_angle_error: 6.4803
Epoch 28: val_mean_angle_error did not improve from 5.85157

Epoch 28: ReduceLROnPlateau reducing learning rate to 1.249999968422344e-05.
Procjena vremena po epohi: 3.4 min; preostalo približno 2.14 h
2653/2653 [==============================] - 203s 76ms/step - loss: 0.0101 - mean_angle_error: 6.4814 - val_loss: 0.0086 - val_mean_angle_error: 6.1539 - lr: 2.5000e-05
Epoch 29/66
2652/2653 [============================>.] - ETA: 0s - loss: 0.0101 - mean_angle_error: 6.4801
Epoch 29: val_mean_angle_error did not improve from 5.85157
Procjena vremena po epohi: 3.4 min; preostalo približno 2.08 h
2653/2653 [==============================] - 203s 76ms/step - loss: 0.0101 - mean_angle_error: 6.4800 - val_loss: 0.0085 - val_mean_angle_error: 6.1087 - lr: 1.2500e-05
Epoch 30/66
2652/2653 [============================>.] - ETA: 0s - loss: 0.0102 - mean_angle_error: 6.4814
Epoch 30: val_mean_angle_erro

Epoch 16/66
2652/2653 [============================>.] - ETA: 0s - loss: 0.0101 - mean_angle_error: 6.4820
Epoch 17: val_mean_angle_error did not improve from 5.71039

Epoch 17: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.
Procjena vremena po epohi: 3.4 min; preostalo približno 2.75 h
2653/2653 [==============================] - 200s 75ms/step - loss: 0.0101 - mean_angle_error: 6.4818 - val_loss: 0.0085 - val_mean_angle_error: 6.1220 - lr: 5.0000e-05
Epoch 18/66
2185/2653 [=======================>......] - ETA: 33s - loss: 0.0102 - mean_angle_error: 6.5340

... (2 outputs left out)

Procjena vremena po epohi: 3.4 min; preostalo približno 2.71 h
2653/2653 [==============================] - 207s 78ms/step - loss: 0.0101 - mean_angle_error: 6.4808 - val_loss: 0.0084 - val_mean_angle_error: 6.0215 - lr: 5.0000e-05
Epoch 19/66
2652/2653 [============================>.] - ETA: 0s - loss: 0.0101 - mean_angle_error: 6.4815
Epoch 19: val_mean_angle_error did not improve from 5.76508
Procjena vremena po epohi: 3.4 min; preostalo približno 2.66 h
2653/2653 [==============================] - 202s 76ms/step - loss: 0.0101 - mean_angle_error: 6.4823 - val_loss: 0.0087 - val_mean_angle_error: 6.2688 - lr: 2.5000e-05
Epoch 20/66
2652/2653 [============================>.] - ETA: 0s - loss: 0.0101 - mean_angle_error: 6.4804
Epoch 20: val_mean_angle_error did not improve from 5.76508
Restoring model weights from the end of the best epoch: 12.
Procjena vremena po epohi: 3.4 min; preostalo približno 2.60 h
2653/2653 [==============================] - 202s 76ms/step - loss: 0.0101 - me

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



2652/2653 [============================>.] - ETA: 0s - loss: 0.0101 - mean_angle_error: 6.4802
Epoch 18: val_mean_angle_error did not improve from 5.88186
Procjena vremena po epohi: 3.4 min; preostalo približno 2.70 h
2653/2653 [==============================] - 205s 77ms/step - loss: 0.0101 - mean_angle_error: 6.4806 - val_loss: 0.0083 - val_mean_angle_error: 5.9393 - lr: 2.5000e-05
Epoch 19/66
2652/2653 [============================>.] - ETA: 0s - loss: 0.0102 - mean_angle_error: 6.4833
Epoch 19: val_mean_angle_error did not improve from 5.88186
Restoring model weights from the end of the best epoch: 11.
Procjena vremena po epohi: 3.4 min; preostalo približno 2.64 h
2653/2653 [==============================] - 205s 77ms/step - loss: 0.0102 - mean_angle_error: 6.4823 - val_loss: 0.0085 - val_mean_angle_error: 6.1200 - lr: 2.5000e-05
Epoch 19: early stopping
Seed 7 završen za 1.08 h; test mean angle error=6.299°
=== seed 13: faza 1, encoder frozen, batch=128 ===
Epoch 1/6
2652/2653 [==

Procjena vremena po epohi: 3.4 min; preostalo približno 2.81 h
2653/2653 [==============================] - 200s 75ms/step - loss: 0.0101 - mean_angle_error: 6.4815 - val_loss: 0.0081 - val_mean_angle_error: 5.7959 - lr: 1.0000e-04
Epoch 17/66
2652/2653 [============================>.] - ETA: 0s - loss: 0.0101 - mean_angle_error: 6.4789
Epoch 17: val_mean_angle_error did not improve from 5.79591
Procjena vremena po epohi: 3.4 min; preostalo približno 2.75 h
2653/2653 [==============================] - 202s 76ms/step - loss: 0.0101 - mean_angle_error: 6.4790 - val_loss: 0.0087 - val_mean_angle_error: 6.2603 - lr: 1.0000e-04
Epoch 18/66
2652/2653 [============================>.] - ETA: 0s - loss: 0.0102 - mean_angle_error: 6.4815
Epoch 18: val_mean_angle_error did not improve from 5.79591
Procjena vremena po epohi: 3.4 min; preostalo približno 2.70 h
2653/2653 [==============================] - 206s 78ms/step - loss: 0.0102 - mean_angle_error: 6.4815 - val_loss: 0.0083 - val_mean_angle_e

## 6. Agregacija preko seedova

`summary.json` sadrži testni mean angle error, standardnu devijaciju uzoraka i mean angle error po ispitaniku. Završna agregacija izostavlja seedove koji nisu imali valjan testni rezultat.


In [6]:
summaries = {}

for seed in SEEDS:

    path = (
        OUTPUT_BASE
        / f"seed_{seed}"
        / "summary.json"
    )

    if path.exists():

        try:

            summaries[int(seed)] = json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as exc:

            warnings.warn(
                f"Ne mogu pročitati {path}: {exc}"
            )


if summaries:

    # -----------------------------------------------------
    # Agregacija rezultata preko seedova
    # -----------------------------------------------------

    seed_means = [

        obj["mean_angle_error"]

        for obj in summaries.values()

        if obj.get("mean_angle_error") is not None
    ]


    # -----------------------------------------------------
    # Agregacija po ispitaniku
    # -----------------------------------------------------

    per_subject = {}

    all_subjects = sorted({

        sid

        for obj in summaries.values()

        for sid in obj.get(
            "per_subject_mean_angle_error",
            {}
        )
    })


    for sid in all_subjects:

        vals = [

            obj["per_subject_mean_angle_error"][sid]

            for obj in summaries.values()

            if sid in obj.get(
                "per_subject_mean_angle_error",
                {}
            )

            and obj["per_subject_mean_angle_error"][sid]
            is not None
        ]

        per_subject[sid] = {

            "mean_angle_error_mean": (
                float(np.mean(vals))
                if vals
                else None
            ),

            "mean_angle_error_std": (
                float(np.std(vals, ddof=1))
                if len(vals) > 1
                else None
            ),

            "n_seeds": len(vals)
        }


    # -----------------------------------------------------
    # Završni summary
    # -----------------------------------------------------

    summary_all = {

        "config": {

            "model": "gaze_only",

            "architecture": (
                "MobileNetV3Small + GAP + Dense(256) "
                "+ Dropout(0.3) + Dense(3) + L2"
            ),

            "loss": "1 - cosine_similarity",

            "metric_unit": "degrees",

            "dataset": str(PREPROCESSED_ROOT)
        },

        "seeds": sorted(summaries),

        "mean_angle_error_mean": (
            float(np.mean(seed_means))
            if seed_means
            else None
        ),

        "mean_angle_error_std": (
            float(np.std(seed_means, ddof=1))
            if len(seed_means) > 1
            else 0.0
        ),

        "per_seed": {

            str(seed): {

                "mean_angle_error": (
                    summaries[seed].get(
                        "mean_angle_error"
                    )
                ),

                "std_angle_error": (
                    summaries[seed].get(
                        "std_angle_error"
                    )
                ),

                "num_subjects": (
                    summaries[seed].get(
                        "num_subjects"
                    )
                )
            }

            for seed in summaries
        },

        "per_subject_mean_angle_error": per_subject,

        "num_subjects_test": max(

            (
                obj.get("num_subjects", 0)
                for obj in summaries.values()
            ),

            default=0
        )
    }


    # -----------------------------------------------------
    # Spremanje summary.json
    # -----------------------------------------------------

    (
        OUTPUT_BASE / "summary.json"
    ).write_text(

        json.dumps(
            summary_all,
            indent=2
        ),

        encoding="utf-8"
    )


    # -----------------------------------------------------
    # Spremanje summary.csv
    # -----------------------------------------------------

    rows = []

    for seed, obj in sorted(summaries.items()):

        rows.append({

            "config": "gaze_only",

            "seed": seed,

            "mean_angle": obj.get(
                "mean_angle_error"
            ),

            "std_angle": obj.get(
                "std_angle_error"
            ),

            "broj_ispitanika": obj.get(
                "num_subjects"
            )
        })


    summary_df = pd.DataFrame(

        rows,

        columns=[
            "config",
            "seed",
            "mean_angle",
            "std_angle",
            "broj_ispitanika"
        ]
    )


    summary_df.to_csv(

        OUTPUT_BASE / "summary.csv",

        index=False
    )


    print(
        summary_df.to_string(index=False)
    )


    if seed_means:

        print(
            f"\nMean ± std preko seedova: "
            f"{summary_all['mean_angle_error_mean']:.4f}° ± "
            f"{summary_all['mean_angle_error_std']:.4f}°"
        )


else:

    print(
        "Nema dovršenih seedova za agregaciju."
    )

   config  seed  mean_angle  std_angle  broj_ispitanika
gaze_only     7    6.299191   4.852648               11
gaze_only    13    6.506437   4.899225               11
gaze_only    42    6.469922   4.894787               11
gaze_only   123    6.465830   4.881349               11
gaze_only  2024    6.494274   4.894207               11

Mean ± std preko seedova: 6.4471° ± 0.0844°
